# Phase 1: Data Integration & Quality (MVP Scope)

## Mục tiêu (Objective)
Thực hiện yêu cầu **3.1 (Chuẩn bị và tích hợp dữ liệu)** của đề bài. 
Chiến lược của chúng ta là không gom tất cả vào 1 bảng khổng lồ (tránh lỗi nhân bản Cartesian). Thay vào đó, chúng ta xây dựng 2 **Data Marts** chuyên biệt để phục vụ trực tiếp cho Dashboard:
1. `datamart_sales.csv`: Hỗ trợ phân tích độ trễ đơn hàng (Q1).
2. `datamart_inventory.csv`: Hỗ trợ phân tích rủi ro vật tư (Q2).

## Table of Contents
1. [Nạp Dữ liệu (Load MVP Domains)](#1-nap-du-lieu)
2. [Data Quality: Chuẩn hóa Kiểu dữ liệu](#2-data-quality)
3. [Tích hợp 1: Xây dựng Sales Datamart (Đánh giá tiến độ)](#3-sales-datamart)
4. [Tích hợp 2: Xây dựng Inventory Datamart (Đánh giá rủi ro)](#4-inventory-datamart)
5. [Xuất dữ liệu cho Streamlit Dashboard](#5-export)


## 1. Nạp Dữ liệu (Load MVP Domains)
Tập trung vào 3 phân hệ cốt lõi nhất để giải quyết bài toán giao hàng và tồn kho.


In [3]:
import pandas as pd
import numpy as np
import os

# Tạo thư mục processed nếu chưa có
os.makedirs("../data/processed", exist_ok=True)

# 1. NẠP DỮ LIỆU TỪ CÁC FILE RIÊNG LẺ TRONG DATA/RAW (MVP Scope)
print("Đang xử lý Sales Domain...")
sales_orders = pd.read_excel("../data/raw/ERP_Sales_Data.xlsx", sheet_name="SalesOrders")
order_lines = pd.read_excel("../data/raw/ERP_Sales_Data.xlsx", sheet_name="OrderLines")

print("Đang xử lý Production Domain...")
prod_orders = pd.read_excel("../data/raw/ERP_Production_Data.xlsx", sheet_name="ProductionOrders")

print("Đang xử lý Materials Domain...")
materials = pd.read_excel("../data/raw/ERP_Materials_Data.xlsx", sheet_name="Materials")
suppliers = pd.read_excel("../data/raw/ERP_Materials_Data.xlsx", sheet_name="Suppliers")

print("Đã nạp thành công 3 Domain: Sales, Production, Materials")


Đang xử lý Sales Domain...
Đang xử lý Production Domain...
Đang xử lý Materials Domain...
Đã nạp thành công 3 Domain: Sales, Production, Materials


## Định nghĩa các hàm hỗ trợ

In [8]:
def check_missing(df_dict):
    """Kiểm tra tỷ lệ Missing Values của từng DataFrame"""
    print("1. CHECK MISSING VALUES:")
    for name, df in df_dict.items():
        missing = df.isnull().sum()
        missing = missing[missing > 0]
        if not missing.empty:
            print(f" ⚠️ {name}:")
            for col, val in missing.items():
                print(f"    - {col}: {val} Nulls ({val/len(df):.1%})")
        else:
            print(f" ✅ {name}: No missing values")
    print("-" * 50)

def check_duplicate(df_dict):
    """Kiểm tra dữ liệu trùng lắp toàn dòng"""
    print("2. CHECK DUPLICATES:")
    for name, df in df_dict.items():
        dups = df.duplicated().sum()
        if dups > 0:
            print(f" ❌ {name}: {dups} duplicated rows!")
        else:
            print(f" ✅ {name}: No duplicates")
    print("-" * 50)

def check_primary_key(df, df_name, pk_col):
    """Kiểm tra tính duy nhất (Uniqueness) của Khóa chính (Primary Key)"""
    is_unique = df[pk_col].is_unique
    has_null = df[pk_col].isnull().any()
    if is_unique and not has_null:
        print(f" ✅ {df_name}[{pk_col}]: Valid Primary Key")
    else:
        print(f" ❌ {df_name}[{pk_col}]: INVALID (Unique: {is_unique}, Has Null: {has_null})")

def check_foreign_key(fact_df, fact_name, fk_col, dim_df, pk_col):
    """Kiểm tra tính vẹn toàn tham chiếu (Referential Integrity) giữa Khóa ngoại và Khóa chính"""
    # Lọc bỏ Null ở Khóa ngoại (vì Null có thể là logic nghiệp vụ hợp lệ)
    valid_fks = fact_df[fk_col].dropna()
    orphans = valid_fks[~valid_fks.isin(dim_df[pk_col])]
    if orphans.empty:
         print(f" ✅ {fact_name}[{fk_col}] -> map 100% to [{pk_col}]")
    else:
         print(f" ❌ {fact_name}[{fk_col}]: Found {len(orphans)} orphan records missing in dimension table!")

def business_rule_validation(df_name, condition, rule_desc):
    """Kiểm tra các quy tắc nghiệp vụ (Business Logic)"""
    violations = condition.sum()
    if violations == 0:
        print(f" ✅ {df_name}: Pass '{rule_desc}'")
    else:
        print(f" ⚠️ {df_name}: {violations} violations found for '{rule_desc}'")


## 2. Data Quality: Chuẩn hóa Kiểu dữ liệu

- `Giả định #1:` Các cột ngày tháng đang ở định dạng Text, cần ép kiểu về Datetime để có thể so sánh và tính toán độ trễ.

-> Dữ liệu từ hệ thống ERP xuất ra file Excel đã được chuẩn hóa định dạng Date từ gốc. Hàm read_excel đã tự động nhận diện thành công kiểu datetime64[ns]. Do đó, chúng ta không cần tốn chi phí tính toán (compute cost) để ép kiểu lại nữa, mà có thể bước thẳng sang khâu tính toán độ trễ!


In [6]:
# 1. SHOW BẰNG CHỨNG TRƯỚC KHI SỬA
print("--- TRƯỚC KHI XỬ LÝ ---")
print(sales_orders[['OrderDate', 'RequiredDate']].dtypes)
print("-" * 30)

# 2. THỰC HIỆN CHUYỂN ĐỔI (Convert to Datetime)
# Sales Dates
for col in ['OrderDate', 'RequiredDate']:
    sales_orders[col] = pd.to_datetime(sales_orders[col], errors='coerce')

# Production Dates
for col in ['PlannedStartDate', 'PlannedEndDate', 'ActualStartDate', 'ActualEndDate']:
    prod_orders[col] = pd.to_datetime(prod_orders[col], errors='coerce')
    
# 3. KẾT QUẢ SAU KHI SỬA
print("--- SAU KHI XỬ LÝ ---")
print(sales_orders[['OrderDate', 'RequiredDate']].dtypes)
print("✅ Chuyển đổi thành công sang kiểu datetime64[ns]!")


--- TRƯỚC KHI XỬ LÝ ---
OrderDate       datetime64[ns]
RequiredDate    datetime64[ns]
dtype: object
------------------------------
--- SAU KHI XỬ LÝ ---
OrderDate       datetime64[ns]
RequiredDate    datetime64[ns]
dtype: object
✅ Chuyển đổi thành công sang kiểu datetime64[ns]!


## 2.5. Data Quality Check (Missing & Outliers)
Trước khi tiến hành hợp nhất (Join) dữ liệu để tạo Data Marts, chúng ta phải đảm bảo các Khóa ngoại (Foreign Keys) và Cột tính toán (Metrics) không chứa dữ liệu bẩn.


In [9]:
# Gom các bảng thành Dictionary để loop
core_dfs = {
    'SalesOrders': sales_orders,
    'OrderLines': order_lines,
    'ProductionOrders': prod_orders,
    'Materials': materials
}

# 1 & 2: Chạy kiểm tra Missing & Duplicates tổng thể
check_missing(core_dfs)
check_duplicate(core_dfs)

# 3. Chạy kiểm tra Primary Keys
print("3. CHECK PRIMARY KEYS:")
check_primary_key(sales_orders, 'SalesOrders', 'OrderID')
check_primary_key(order_lines, 'OrderLines', 'OrderLineID')
check_primary_key(prod_orders, 'ProductionOrders', 'ProductionOrderID')
print("-" * 50)

# 4. Chạy kiểm tra Foreign Keys (Tính vẹn toàn tham chiếu)
print("4. CHECK FOREIGN KEYS (Referential Integrity):")
check_foreign_key(order_lines, 'OrderLines', 'OrderID', sales_orders, 'OrderID')
check_foreign_key(prod_orders, 'ProductionOrders', 'SalesOrderID', sales_orders, 'OrderID')
print("-" * 50)

# 5. Chạy kiểm tra Business Rules (Ví dụ: Số lượng không được âm)
print("5. BUSINESS RULE VALIDATION:")
business_rule_validation('OrderLines', order_lines['Quantity'] <= 0, "Quantity must be strictly positive")
business_rule_validation('Materials', materials['CurrentStock'] < 0, "Stock cannot be negative")


1. CHECK MISSING VALUES:
 ✅ SalesOrders: No missing values
 ✅ OrderLines: No missing values
 ⚠️ ProductionOrders:
    - SalesOrderID: 22 Nulls (27.5%)
    - ActualStartDate: 14 Nulls (17.5%)
    - ActualEndDate: 28 Nulls (35.0%)
 ✅ Materials: No missing values
--------------------------------------------------
2. CHECK DUPLICATES:
 ✅ SalesOrders: No duplicates
 ✅ OrderLines: No duplicates
 ✅ ProductionOrders: No duplicates
 ✅ Materials: No duplicates
--------------------------------------------------
3. CHECK PRIMARY KEYS:
 ✅ SalesOrders[OrderID]: Valid Primary Key
 ✅ OrderLines[OrderLineID]: Valid Primary Key
 ✅ ProductionOrders[ProductionOrderID]: Valid Primary Key
--------------------------------------------------
4. CHECK FOREIGN KEYS (Referential Integrity):
 ✅ OrderLines[OrderID] -> map 100% to [OrderID]
 ✅ ProductionOrders[SalesOrderID] -> map 100% to [OrderID]
--------------------------------------------------
5. BUSINESS RULE VALIDATION:
 ✅ OrderLines: Pass 'Quantity must be s

## 3. Tích hợp 1: Xây dựng Sales Datamart (Tiến độ & Giao hàng)
Mục tiêu là nối `SalesOrders` với doanh thu thực tế từ `OrderLines`, và lấy ngày xuất xưởng thực tế từ `ProductionOrders` để gán cờ rủi ro **Trễ hẹn (Is_Delayed)**.


In [ ]:
# 1. Tính doanh thu chuẩn xác từ OrderLines (tránh nhân bản)
order_lines['LineTotal_Calc'] = order_lines['Quantity'] * order_lines['UnitPrice']
revenue_df = order_lines.groupby('OrderID', as_index=False)['LineTotal_Calc'].sum()
datamart_sales = pd.merge(sales_orders, revenue_df, on='OrderID', how='left')

# 2. Lấy ngày hoàn thành sản xuất trễ nhất để gán cho Đơn hàng
# (Các lệnh Make-to-stock SalesOrderID=Null đã tự động rớt ra): xóa lệnh sản xuất tồn kho khỏi quá trình join này vì nó ko thuộc về bất kỳ đơn hàng nào cả 
prod_summary = prod_orders.dropna(subset=['SalesOrderID']).groupby('SalesOrderID', as_index=False).agg({
    'ActualEndDate': 'max', 
})
prod_summary.rename(columns={'SalesOrderID': 'OrderID'}, inplace=True)
datamart_sales = pd.merge(datamart_sales, prod_summary, on='OrderID', how='left')

# 3. Logic tính Trễ hạn
# xử lí trễ hẹn nếu vượt quá ngày cam kết giao hàng thì đánh cờ delay = true luôn, 
# Giả lập ngày "hiện tại" là ngày xa nhất trong hệ thống
today_mock = datamart_sales['RequiredDate'].max() 

# Điều kiện 1: Đã giao nhưng muộn hơn cam kết
is_late_delivered = datamart_sales['ActualEndDate'] > datamart_sales['RequiredDate']
# Điều kiện 2: Chưa giao (ActualEndDate Null) và thời gian hiện tại đã lố hạn cam kết
is_overdue_not_done = datamart_sales['ActualEndDate'].isnull() & (datamart_sales['RequiredDate'] < today_mock)

datamart_sales['Is_Delayed'] = is_late_delivered | is_overdue_not_done
datamart_sales['Risk_Level'] = np.where(datamart_sales['Is_Delayed'], 'High', 'Low')

display(datamart_sales[['OrderID', 'OrderDate', 'RequiredDate', 'ActualEndDate', 'Is_Delayed', 'LineTotal_Calc']].head(3))


,OrderID,OrderDate,RequiredDate,ActualEndDate,Is_Delayed,LineTotal_Calc
0,SO-00001,2026-01-13,2026-02-13,NaT,True,56115
1,SO-00002,2026-02-17,2026-03-16,NaT,True,307975
2,SO-00003,2026-03-23,2026-04-21,NaT,True,37625


## 4. Tích hợp 2: Xây dựng Inventory Datamart (Rủi ro vật tư)
Nối bảng `Materials` với `Suppliers` để tìm các mặt hàng đang cạn kho (dưới mức Reorder Point).


In [11]:
datamart_inventory = pd.merge(materials, suppliers[['SupplierID', 'SupplierName', 'LeadTimeAvg']], on='SupplierID', how='left')

# Cờ rủi ro đứt hàng: Nếu Tồn kho hiện tại thấp hơn mức Cảnh báo đặt hàng
datamart_inventory['Stockout_Risk'] = datamart_inventory['CurrentStock'] < datamart_inventory['ReorderPoint']

# Hiển thị những vật tư đang cạn kho
display(datamart_inventory[datamart_inventory['Stockout_Risk'] == True][['MaterialID', 'MaterialName', 'CurrentStock', 'ReorderPoint', 'LeadTimeAvg']].head(3))


,MaterialID,MaterialName,CurrentStock,ReorderPoint,LeadTimeAvg
11,MAT-0012,Carbon Steel,157,482,28
16,MAT-0017,Rubber Seal,151,372,18
34,MAT-0035,Powder Coating,277,393,7


## 5. Xuất dữ liệu (Export cho Streamlit)


In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)

datamart_sales.to_csv("../data/processed/datamart_sales.csv", index=False)
datamart_inventory.to_csv("../data/processed/datamart_inventory.csv", index=False)

print(f"✅ Đã tạo datamart_sales.csv với {len(datamart_sales)} dòng")
print(f"✅ Đã tạo datamart_inventory.csv với {len(datamart_inventory)} dòng")
print("Đã sẵn sàng dữ liệu cho Streamlit Dashboard!")


✅ Đã tạo datamart_sales.csv với 100 dòng
✅ Đã tạo datamart_inventory.csv với 50 dòng
Đã sẵn sàng dữ liệu cho Streamlit Dashboard!


## 6. Kiểm tra 2 bộ Datamarts vừa thu được

In [13]:
import sys
# Nạp thư viện viz_utils để hiển thị báo cáo xịn xò
sys.path.insert(0, '../src')
from viz_utils import *
setup()

# Nạp lại chính file CSV vừa xuất để mô phỏng chính xác những gì Phase 2 sẽ nhận
final_sales = pd.read_csv("../data/processed/datamart_sales.csv")
final_inv = pd.read_csv("../data/processed/datamart_inventory.csv")

section("📦 HẬU KIỂM 1: DATAMART SALES", "Đảm bảo không bị nhân bản (Cartesian explosion) sau khi Join")
# Nếu Rows = 100 là thành công tuyệt đối!
quick_profile(final_sales)

section("📦 HẬU KIỂM 2: DATAMART INVENTORY", "Kiểm tra sự toàn vẹn của bảng Tồn kho")
# Nếu Rows = 50 là thành công tuyệt đối!
quick_profile(final_inv)


,dtype,non_null,null,null_%,unique,sample
OrderID,object,100,0,0.0,100,SO-00001
CustomerID,object,100,0,0.0,19,CUST-0013
OrderDate,object,100,0,0.0,62,2026-01-13
RequiredDate,object,100,0,0.0,59,2026-02-13
OrderStatus,object,100,0,0.0,5,Shipped
Priority,object,100,0,0.0,4,High
SalesRepID,object,100,0,0.0,10,EMP-0003
LineTotal_Calc,int64,100,0,0.0,100,56115
ActualEndDate,object,32,68,68.0,24,NaN
Is_Delayed,bool,100,0,0.0,2,True


,dtype,non_null,null,null_%,unique,sample
MaterialID,object,50,0,0.0,50,MAT-0001
MaterialName,object,50,0,0.0,50,Steel Sheet 4mm
MaterialCategory,object,50,0,0.0,24,Steel
UnitOfMeasure,object,50,0,0.0,11,Sheet
UnitCost,int64,50,0,0.0,27,45
SupplierID,object,50,0,0.0,15,SUP-010
LeadTimeDays,int64,50,0,0.0,17,7
MinOrderQty,int64,50,0,0.0,5,100
CurrentStock,int64,50,0,0.0,49,372
ReorderPoint,int64,50,0,0.0,47,254
